# ②.B 第二階段：模型訓練與預測模組 (進階) (Model Training & Prediction – Advanced)

本 Notebook 專注於專案計畫「②.B 第二階段：模型訓練與預測模組 (進階)」的內容。主要任務包含：

1.  **載入資料**: 
    *   載入經過完整特徵工程 (包含基礎與進階特徵) 和標註的資料集。
2.  **2.B.1 模型選擇與特徵設計 (回顧與應用)**:
    *   根據計畫中的討論，選擇合適的進階模型 (如 XGBoost, LightGBM, LSTM, Transformer 等)。
    *   確認使用的特徵集是否包含計畫中提及的「跨時間點」核心特徵 (如均線斜率、VWAP 偏離、前 N 分鐘極值、成交量加權指標、時間區段標籤等)。
3.  **2.B.2 標註欄位設計 (回顧與應用)**:
    *   確認使用的標註欄位符合計畫中的設計原則 (如未來價格漲跌分類、未來報酬率分類、動態上下界、多階段標註等)。
4.  **2.B.3 時序資料切分 (Time-Series Split)**:
    *   實現並應用更嚴謹的時序資料切分策略，例如：
        *   `sklearn.model_selection.TimeSeriesSplit`。
        *   (選做) 跨週交叉驗證 (Cross-Week CV)。
        *   (選做) 滾動窗口訓練 (Rolling-Window Training)。
5.  **模型訓練與調優**:
    *   訓練選擇的進階模型。
    *   (選做) 進行超參數調優 (e.g., using `GridSearchCV`, `RandomizedSearchCV`, or Bayesian Optimization)。
6.  **2.B.4 訓練與驗證 Pipeline (參考與擴展)**:
    *   建立或擴展訓練 Pipeline，可能包含資料預處理 (如缺失值填充、標準化)、過採樣/欠採樣 (如 `imblearn.over_sampling.RandomOverSampler`)、模型訓練等步驟。
7.  **模型評估**:
    *   使用適當的評估指標 (如 Accuracy, Precision, Recall, F1-score, ROC-AUC) 對模型在驗證集/測試集上的表現進行詳細評估。
8.  **模型解釋性 (SHAP)**:
    *   (若適用於所選模型，如 Tree-based 模型) 使用 SHAP 或類似工具進行模型解釋性分析，了解特徵重要性及特徵對預測的影響。
9.  **儲存模型**:
    *   儲存訓練好的進階模型以供後續回測與部署使用。

---

In [ ]:
# 初始設定與導入必要函式庫
import pandas as pd
import numpy as np
import os
import sys
import joblib # 用於儲存模型

# 將專案根目錄添加到 Python 搜尋路徑
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 例如：從 utils 或 features 導入需要的模組
# from utils.data_loader import load_processed_data # 假設有函數載入已處理資料
# from features.feature_engineering import add_advanced_features # 假設有函數添加進階特徵
# from features.labeling import generate_label_tick # 或其他標註函數
# from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.impute import SimpleImputer
# from imblearn.over_sampling import RandomOverSampler # 如需處理類別不平衡
# import xgboost as xgb
# # import lightgbm as lgb # 如果使用 LightGBM
# # from tensorflow import keras # 如果使用 LSTM/Transformer
# from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
# import shap

## 1. 載入資料
載入已經過特徵工程和標註的資料集。這裡假設資料已包含所有必要的基礎及進階特徵。

In [ ]:
# data_path_with_all_features = '../data/ticks_2025-0527_with_all_features.parquet' # 假設檔名
# try:
#     df = pd.read_parquet(data_path_with_all_features)
#     print(f"資料 {data_path_with_all_features} 載入成功。")
#     df.dropna(subset=['label'], inplace=True) # 確保標籤存在
#     # 根據需要處理其他 NaN，例如特徵中的 NaN
# except FileNotFoundError:
#     print(f"錯誤：找不到資料檔案 {data_path_with_all_features}。請先執行完整的特徵工程。")
#     df = None

# if df is not None:
#     print(df.head())
#     print(df.info())

## 2. 定義特徵 (X) 與目標 (y)

In [ ]:
# if df is not None:
#     # 根據實際情況選擇特徵欄位
#     # 應排除掉原始價格欄位 (open, high, low, close), volume, amount, ticks, timestamp 及 label 本身
#     # 以及任何在標註過程中可能引入未來資訊的輔助欄位
#     exclude_cols = ['timestamp', 'label', 'open', 'high', 'low', 'volume', 'amount', 'ticks', 'future_price', 'price_diff'] # 示例排除欄位
#     features = [col for col in df.columns if col not in exclude_cols and not col.startswith('label_') and not col.startswith('target_')]
#     target = 'label'

#     X = df[features]
#     y = df[target]

#     print(f"Features ({len(X.columns.tolist())}): {X.columns.tolist()}")
#     print(f"Target: {target}")
#     print(f"Target distribution:\n{y.value_counts(normalize=True)}")

## 3. 時序資料切分 (TimeSeriesSplit)

In [ ]:
# if 'X' in locals() and 'y' in locals():
#     from sklearn.model_selection import TimeSeriesSplit
#     n_splits = 5 # 可根據資料量調整
#     tscv = TimeSeriesSplit(n_splits=n_splits)

#     print(f"Performing TimeSeriesSplit with {n_splits} splits.")
#     # 通常取最後一折進行訓練與測試，或進行完整的交叉驗證流程
#     for i, (train_index, test_index) in enumerate(tscv.split(X)):
#         print(f"Fold {i+1}: Train size={len(train_index)}, Test size={len(test_index)}")
#         if i == n_splits - 1: # 取最後一折
#             X_train, X_test = X.iloc[train_index], X.iloc[test_index]
#             y_train, y_test = y.iloc[train_index], y.iloc[test_index]

#     print(f"\nFinal split: X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

## 4. 建立訓練 Pipeline (例如 XGBoost)
包含預處理 (缺失值填充、標準化)、類別不平衡處理 (可選)、模型訓練。

In [ ]:
# if 'X_train' in locals():
#     import xgboost as xgb
#     from sklearn.pipeline import Pipeline
#     from sklearn.impute import SimpleImputer
#     from sklearn.preprocessing import StandardScaler
#     from imblearn.over_sampling import RandomOverSampler # 處理類別不平衡
#     from imblearn.pipeline import Pipeline as ImbPipeline # imblearn 的 Pipeline

#     # 調整 y_train 和 y_test 的值，使其從 0 開始 (如果原始標籤是 -1, 0, 1)
#     # XGBoost 多分類的標籤需要是 0, 1, 2, ...
#     y_train_xgb = y_train.replace({-1: 0, 0: 1, 1: 2})
#     y_test_xgb = y_test.replace({-1: 0, 0: 1, 1: 2})

#     # 建立 Pipeline
#     xgb_pipeline = ImbPipeline([
#         ('imputer', SimpleImputer(strategy='mean')), # 填充缺失值
#         ('scaler', StandardScaler()), # 特徵標準化
#         ('oversampler', RandomOverSampler(random_state=42)), # 過採樣處理類別不平衡
#         ('xgb', xgb.XGBClassifier(
#             objective='multi:softmax', 
#             num_class=3, 
#             eval_metric='mlogloss', 
#             use_label_encoder=False, 
#             random_state=42,
#             # 其他 XGBoost 參數可以進行調優
#             # n_estimators=100,
#             # learning_rate=0.1,
#             # max_depth=3
#         ))
#     ])

#     print("Training XGBoost pipeline...")
#     # 訓練模型，可以使用 early stopping
#     # XGBoost 的 early stopping 需要在 fit 方法中傳遞 eval_set
#     # Pipeline 中的 XGBoost 需要特殊處理才能使用 early stopping，或者單獨訓練 XGBoost 模型
#     # 這裡我們先直接 fit pipeline
#     xgb_pipeline.fit(X_train, y_train_xgb)
#     print("XGBoost pipeline training complete.")

## 5. 模型評估

In [ ]:
# if 'xgb_pipeline' in locals() and 'X_test' in locals():
#     from sklearn.metrics import classification_report, accuracy_score

#     y_pred_pipeline = xgb_pipeline.predict(X_test)

#     print("\nXGBoost Pipeline Classification Report:")
#     # 注意 target_names 要與 y_test_xgb 的標籤對應
#     print(classification_report(y_test_xgb, y_pred_pipeline, target_names=['Down (Class 0)', 'Neutral (Class 1)', 'Up (Class 2)']))
#     print(f"XGBoost Pipeline Accuracy: {accuracy_score(y_test_xgb, y_pred_pipeline):.4f}")

#     # 如果需要計算 ROC AUC，對於多分類問題需要特別處理
#     # y_proba_pipeline = xgb_pipeline.predict_proba(X_test)
#     # roc_auc = roc_auc_score(y_test_xgb, y_proba_pipeline, multi_class='ovr') # One-vs-Rest
#     # print(f"XGBoost Pipeline ROC AUC (OvR): {roc_auc:.4f}")

## 6. 模型解釋性 (SHAP)
對於 Pipeline 中的模型，提取模型本身進行 SHAP 分析。

In [ ]:
# if 'xgb_pipeline' in locals():
#     import shap
#     import matplotlib.pyplot as plt

#     # 從 Pipeline 中獲取訓練好的 XGBoost 模型
#     trained_xgb_model = xgb_pipeline.named_steps['xgb']

#     # SHAP 分析需要處理過的特徵 (例如經過 imputer 和 scaler)
#     # 這裡我們用 Pipeline 的 transform 方法來獲取處理後的 X_test
#     # 注意：oversampler 只在 fit 時作用，不影響 transform
#     X_test_transformed = xgb_pipeline.named_steps['scaler'].transform(
#         xgb_pipeline.named_steps['imputer'].transform(X_test)
#     )
#     X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=X_test.columns)

#     print("\nGenerating SHAP values...")
#     explainer = shap.TreeExplainer(trained_xgb_model)
#     shap_values = explainer.shap_values(X_test_transformed_df)

#     # 繪製 SHAP 摘要圖 (整體特徵重要性)
#     # shap_values 對於多分類是一個列表，每個元素對應一個類別的 SHAP 值
#     plt.figure()
#     shap.summary_plot(shap_values, X_test_transformed_df, plot_type="bar", class_names=['Down', 'Neutral', 'Up'], show=False)
#     plt.title("SHAP Feature Importance (Overall Classes)")
#     plt.show()

#     # 繪製某個特定類別的 SHAP 摘要圖，例如預測為 'Up' (Class 2)
#     class_index_to_plot = 2 # 假設 'Up' 是第3個類別 (索引為2)
#     plt.figure()
#     shap.summary_plot(shap_values[class_index_to_plot], X_test_transformed_df, show=False)
#     plt.title(f"SHAP Summary Plot for Class: Up (Original Label 1)")
#     plt.show()

## 7. 儲存訓練好的 Pipeline

In [ ]:
# if 'xgb_pipeline' in locals():
#     model_dir = '../trained_models'
#     if not os.path.exists(model_dir):
#         os.makedirs(model_dir)

#     advanced_model_filename = os.path.join(model_dir, 'advanced_xgb_pipeline.joblib')
#     joblib.dump(xgb_pipeline, advanced_model_filename)

#     print(f"\nAdvanced XGBoost pipeline saved to {advanced_model_filename}")